# GraphRAG Retrieval on SciFact

This notebook runs **entity retrieval + graph traversal** from GraphRAG on the SciFact benchmark,
skipping the final LLM synthesis step.

## Prerequisites
```bash
# 1. Prepare SciFact input files (one-time)
cd graphrag_quickstart
python prepare_scifact.py

# 2. Index (one-time, ~10-20 min depending on corpus size)
graphrag index
```

## Pipeline overview
```
SciFact claim  →  entity embedding lookup (LanceDB)  →  graph traversal
               →  candidate text units  →  rank by coverage
               →  map text_unit.document_id → doc_id  →  evaluate
```

**Note:** This notebook must be run in the `.venv` kernel (Python 3.13, graphrag 3.0.9).
Make sure `GRAPHRAG_API_KEY` is set in `.env` (needed for query-time entity embedding).

In [1]:
!pip install graphrag==3.0.9

In [ ]:
import os
import sys
import json
from pathlib import Path

# Paths
NOTEBOOK_DIR  = Path(".").resolve()               # graphrag_quickstart/
REPO_ROOT     = NOTEBOOK_DIR.parent               # ANLP-HW34/
OUTPUT_DIR    = NOTEBOOK_DIR / "output"
LANCEDB_URI   = str(OUTPUT_DIR / "lancedb")
RESULTS_DIR   = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Allow importing from hydeOnSciFact
sys.path.insert(0, str(REPO_ROOT / "hydeOnSciFact"))

print("NOTEBOOK_DIR :", NOTEBOOK_DIR)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("LANCEDB_URI  :", LANCEDB_URI)

NOTEBOOK_DIR : /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3&4/baseline_reproduction/ANLP-HW34/graphrag_hyde
OUTPUT_DIR   : /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3&4/baseline_reproduction/ANLP-HW34/graphrag_hyde/output
LANCEDB_URI  : /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3&4/baseline_reproduction/ANLP-HW34/graphrag_hyde/output/lancedb


## 1. Load SciFact corpus, queries, and relevance labels

In [3]:
from load_data import load_scifact_data

corpus, queries, qrels, data_source = load_scifact_data(split="test")

query_ids      = list(queries.keys())
corpus_doc_ids = list(corpus.keys())

print(f"Source : {data_source}")
print(f"Corpus : {len(corpus):,} docs  |  Queries: {len(queries):,}  |  Qrels: {len(qrels):,}")

/Users/winstondong/miniforge3/envs/adnlp_hyde_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Source : mteb/scifact (corpus:corpus/corpus, queries:queries/queries, qrels:default/test)
Corpus : 5,183 docs  |  Queries: 300  |  Qrels: 300


## 2. Load GraphRAG config and indexed artefacts

In [4]:
import pandas as pd
from graphrag.config.load_config import load_config
from graphrag.query.indexer_adapters import (
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
    read_indexer_communities,
)

# Load settings.yaml → GraphRagConfig
config = load_config(root_dir=NOTEBOOK_DIR)
print("Config loaded. Embedding model:",
      config.embedding_models["default_embedding_model"].model)

# Load parquet artefacts produced by `graphrag index`
entities_df   = pd.read_parquet(OUTPUT_DIR / "entities.parquet")
communities_df = pd.read_parquet(OUTPUT_DIR / "communities.parquet")
reports_df    = pd.read_parquet(OUTPUT_DIR / "community_reports.parquet")
text_units_df = pd.read_parquet(OUTPUT_DIR / "text_units.parquet")
rels_df       = pd.read_parquet(OUTPUT_DIR / "relationships.parquet")
documents_df  = pd.read_parquet(OUTPUT_DIR / "documents.parquet")

COMMUNITY_LEVEL = 0  # adjust if your graph has fewer levels

entities      = read_indexer_entities(entities_df, communities_df, COMMUNITY_LEVEL)
relationships = read_indexer_relationships(rels_df)
text_units    = read_indexer_text_units(text_units_df)
reports       = read_indexer_reports(reports_df, communities_df, COMMUNITY_LEVEL)

print(f"Entities: {len(entities):,}  |  Text units: {len(text_units):,}  "
      f"|  Relationships: {len(relationships):,}  |  Reports: {len(reports):,}")

Config loaded. Embedding model: text-embedding-3-large
Entities: 19,919  |  Text units: 5,555  |  Relationships: 35,372  |  Reports: 0


## 3. Build helper lookups for the doc-ID mapping

After indexing, each file `<doc_id>.txt` becomes a row in `documents.parquet`
with `title = "<doc_id>.txt"`. Text units carry a `document_id` UUID that
points back to that row. We build two dicts to follow the chain:

```
text_unit.short_id  →  text_unit.document_id (UUID)
                    →  document title ("<doc_id>.txt")
                    →  scifact_doc_id (strip ".txt")
```

In [5]:
# short_id (human_readable_id) → TextUnit object
unit_by_short_id: dict = {u.short_id: u for u in text_units}

# document UUID → SciFact doc_id string
doc_uuid_to_scifact_id: dict = {
    row["id"]: str(row["title"]).removesuffix(".txt")
    for _, row in documents_df.iterrows()
}

def text_unit_short_ids_to_doc_ids(short_ids: list[str]) -> list[str]:
    """Convert an ordered list of text-unit short_ids to deduplicated SciFact doc IDs."""
    seen = set()
    result = []
    for sid in short_ids:
        unit = unit_by_short_id.get(sid)
        if unit is None or unit.document_id is None:
            continue
        scifact_id = doc_uuid_to_scifact_id.get(unit.document_id)
        if scifact_id and scifact_id not in seen:
            seen.add(scifact_id)
            result.append(scifact_id)
    return result

print(f"Lookup tables ready. documents_df shape: {documents_df.shape}")
print("Sample doc titles:", documents_df["title"].head(3).tolist())

Lookup tables ready. documents_df shape: (5183, 7)
Sample doc titles: ['17897801.txt', '8519911.txt', '4418070.txt']


## 4. Connect to LanceDB and build the context engine

We use GraphRAG's `get_local_search_engine` factory — it wires together the
embedding model, the entity vector store, and the `LocalSearchMixedContext`
context builder from `settings.yaml` in one call.

We then call `context_builder.build_context()` directly, which performs
**entity retrieval + graph traversal** but never calls the completion model.

In [6]:
from graphrag_vectors.lancedb import LanceDBVectorStore
from graphrag.config.embeddings import entity_description_embedding
from graphrag.query.factory import get_local_search_engine

# Connect to the entity-description vector store created during indexing
entity_embedding_store = LanceDBVectorStore(
    index_name=entity_description_embedding,  # table = "entity_description"
    db_uri=LANCEDB_URI,
)
entity_embedding_store.connect()
print("LanceDB connected. Table:", entity_description_embedding)

# Build the LocalSearch engine (completion model is wired but never called
# because we stop before the .search() step)
search_engine = get_local_search_engine(
    config=config,
    reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    covariates={},
    response_type="single paragraph",
    description_embedding_store=entity_embedding_store,
)

context_builder = search_engine.context_builder
print("LocalSearchMixedContext ready.")

LanceDB connected. Table: entity_description
LocalSearchMixedContext ready.


## 5. Retrieve — call context_builder only (no LLM synthesis)

`build_context()` returns a `ContextBuilderResult` whose `context_records`
dict contains a `"sources"` DataFrame with columns `["id", "text"]` where
`"id"` is the text-unit `short_id` (human_readable_id).  
We map those back to SciFact corpus IDs through the lookup tables built above.

In [7]:
from tqdm import tqdm

TOP_K         = 10   # max docs to retrieve per query
TOP_KS        = [1, 3, 5, 10]
MAX_CTX_TOKENS = 8000

per_query_retrieved: dict[str, list[str]] = {}
per_query_rows: list[dict] = []

def extract_entities_info(df) -> list[dict]:
    if df is None or df.empty:
        return []
    cols = df.columns.tolist()
    name_col = next((c for c in ["entity", "name", "title"] if c in cols), None)
    desc_col = "description" if "description" in cols else None
    score_col = next((c for c in ["rank", "score", "weight"] if c in cols), None)
    result = []
    for _, row in df.iterrows():
        item = {}
        if name_col:  item["name"]  = row[name_col]
        if desc_col:  item["description"] = row[desc_col]
        if score_col: item["score"] = row[score_col]
        result.append(item)
    return result

def extract_rels_info(df) -> list[dict]:
    if df is None or df.empty:
        return []
    cols = df.columns.tolist()
    result = []
    for _, row in df.iterrows():
        item = {}
        for c in ["source", "target", "description", "weight", "rank"]:
            if c in cols:
                item[c] = row[c]
        result.append(item)
    return result

for qid in tqdm(query_ids, desc="Retrieving"):
    claim = queries[qid]

    ctx_result = context_builder.build_context(
        query=claim,
        max_context_tokens=MAX_CTX_TOKENS,
    )

    records = ctx_result.context_records

    sources_df = records.get("sources", None)
    entities_df_ctx = records.get("entities", None)
    rels_df_ctx = records.get("relationships", None)

    if sources_df is None or sources_df.empty:
        retrieved_doc_ids: list[str] = []
    else:
        retrieved_doc_ids = text_unit_short_ids_to_doc_ids(
            sources_df["id"].tolist()
        )[:TOP_K]

    per_query_retrieved[qid] = retrieved_doc_ids

    per_query_rows.append({
        "query_id"         : qid,
        "query"            : claim,
        "gold_doc_ids"     : sorted(qrels[qid]),
        "retrieved_doc_ids": retrieved_doc_ids,
        "matched_entities" : extract_entities_info(entities_df_ctx),
        "matched_rels"     : extract_rels_info(rels_df_ctx),
        "hit@1"            : int(any(d in qrels[qid] for d in retrieved_doc_ids[:1])),
        "hit@5"            : int(any(d in qrels[qid] for d in retrieved_doc_ids[:5])),
        "hit@10"           : int(any(d in qrels[qid] for d in retrieved_doc_ids[:10])),
    })

print(f"Retrieval done for {len(per_query_retrieved)} queries.")
# Preview first result
if per_query_rows:
    r = per_query_rows[0]
    print(f"\nQuery: {r['query']}")
    print(f"Entities ({len(r['matched_entities'])}): {[e.get('name') for e in r['matched_entities'][:5]]}")
    print(f"Rels    ({len(r['matched_rels'])}): {[(e.get('source'), e.get('target')) for e in r['matched_rels'][:3]]}")


Retrieving: 100%|██████████| 300/300 [02:15<00:00,  2.21it/s]

Retrieval done for 300 queries.

Query: 0-dimensional biomaterials show inductive properties.
Entities (15): ['NANOMETER-SCALE SCAFFOLDS', 'GRAPHENE', 'HYDROGELS', 'MICROMOLDED ELASTOMERIC MICROPOST ARRAYS', 'COMPLEX MULTICELLULAR STRUCTURES']
Rels    (23): [('3D MIGRATION AND DIFFERENTIATION ASSAY', 'LAYERED HYDROGELS'), ('HUMAN EMBRYONIC STEM CELLS', 'HYDROGEL-BASED COMPLIANT MATRIX'), ('HUMAN EMBRYONIC STEM CELLS', 'STIFF HYDROGEL MATRIX')]


## 6. Evaluate

In [10]:
import sys
sys.path.insert(0, str(Path("..") / "hydeOnSciFact"))
from evaluate import evaluate_run


metrics = evaluate_run(per_query_retrieved, qrels, TOP_KS)

print("\n=== GraphRAG Retrieval — SciFact ===\n")
for k in ["Recall@1", "Recall@5", "Recall@10", "MRR@10", "nDCG@10"]:
    if k in metrics:
        print(f"  {k:<12}: {metrics[k]:.4f}")


=== GraphRAG Retrieval — SciFact ===

  Recall@1    : 0.4429
  Recall@5    : 0.6346
  Recall@10   : 0.7066
  MRR@10      : 0.5430
  nDCG@10     : 0.5780


## 7. Save results

In [11]:
metrics_payload = {
    "config": {
        "mode"          : "graphrag_local_search",
        "split"         : "test",
        "data_source"   : data_source,
        "community_level": COMMUNITY_LEVEL,
        "max_ctx_tokens": MAX_CTX_TOKENS,
        "top_ks"        : TOP_KS,
    },
    "metrics": metrics,
}

metrics_path = RESULTS_DIR / "metrics.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

per_query_path = RESULTS_DIR / "per_query_results.jsonl"
with per_query_path.open("w", encoding="utf-8") as f:
    for row in per_query_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved metrics     : {metrics_path}")
print(f"Saved per-query   : {per_query_path}")

# Print entity/rel coverage stats
n_with_entities = sum(1 for r in per_query_rows if r["matched_entities"])
n_with_rels     = sum(1 for r in per_query_rows if r["matched_rels"])
print(f"\nQueries with matched_entities : {n_with_entities} / {len(per_query_rows)}")
print(f"Queries with matched_rels     : {n_with_rels} / {len(per_query_rows)}")


Saved metrics     : /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3&4/baseline_reproduction/ANLP-HW34/graphrag_hyde/results/metrics.json
Saved per-query   : /Users/winstondong/CMUCourses/AdvancedNLP/Assignment_3&4/baseline_reproduction/ANLP-HW34/graphrag_hyde/results/per_query_results.jsonl

Queries with matched_entities : 300 / 300
Queries with matched_rels     : 299 / 300
